# BALANCEAMENTO DE TRÁFEGO

In [1]:
import pandas as pd
import numpy as np

#### IMPORTAR AS BASES (INPUTS)

In [2]:
#Import Informações da Planilha B.4021
basekpi = pd.read_excel('Inputs/4021-LTE- 2ºHMM Volume Dados - Semanal.xlsx', usecols=[0,1,2,5,7,10,12,14])

In [3]:
#Import Informações da Planilha Relatório de Ocupação de Células 4G – Semanal
baseocup = pd.read_excel('Inputs/Relatório de Ocupação de Células 4G – Semanal.xlsx', usecols=[1,2,3,4,6,7,8,9,15])
#Filtrar Regional TNE
baseocup = baseocup[baseocup.Regional.eq('TNE')]
baseocup = baseocup[['Estado', 'ANF', 'Município', 'Banda', 'BTS/NodeB/ENodeB','Célula', 'Classificação', 'Classificação Populacional']]

In [4]:
#Import Informações da Planilha PRB AVAIL
basebw = pd.read_excel('Inputs/Relatório Personalizado LTE - Diário - PRB AVAIL.xlsx', usecols=[0,1,4])

In [5]:
#Import Informações da Planilha Spazio
spazio = pd.read_excel('Inputs/Spazio.xlsx', usecols=[0,4,27])
spazio = spazio[spazio['CEP'] != "0"]
spazio.drop_duplicates(subset=['Endereço ID'],keep="first" , inplace= True)   

#### UNINDO AS BASES

In [6]:
base = pd.merge(basekpi,baseocup, left_on='Célula', right_on='Célula', how='right')

#### Determinar largura de banda

In [7]:
basebw = basebw.groupby(['Célula'], sort=False)['PRB_AVAIL'].max()
basebw= basebw.to_frame()

In [8]:
def PRB(basebw):

    if (basebw['PRB_AVAIL'] == 0) :
        return 'Indefinido'
    elif (basebw['PRB_AVAIL'] > 0 and basebw['PRB_AVAIL'] <= 37.5):
        return '5'
    elif (basebw['PRB_AVAIL'] > 37.5 and basebw['PRB_AVAIL'] <= 62.5):
        return '10'
    elif (basebw['PRB_AVAIL'] > 62.5 and basebw['PRB_AVAIL'] <= 84.5):
        return '15'
    elif (basebw['PRB_AVAIL'] > 84.5 and basebw['PRB_AVAIL'] <= 112.5):
        return '20'
    elif (basebw['PRB_AVAIL'] > 112.5):
        return '25'

basebw['PRB_AVAIL'] = basebw.apply(PRB, axis=1)

In [9]:
basebw.rename(columns={'PRB_AVAIL': 'BW_MHz'}, inplace=True)

In [10]:
base = pd.merge(base,basebw, left_on='Célula', right_on='Célula', how='left')
base = pd.merge(base,spazio, left_on='BTS/NodeB/ENodeB', right_on='Site ID', how='left')
base = base[['Semana do Ano', 'Estado', 'ANF', 'Município', 'Classificação Populacional', 'Endereço ID', 'Célula', 'Fornecedor', 'Banda', 'BW_MHz', 'Classificação','BTS/NodeB/ENodeB','PRB_UTIL_MEAN_DL','AVG_USERS_RRC_CONN_MEAN_SUM','AVG_THROU_PDCP_USER_DL','VOLUME_DADOS_DL_ALLOP 4G','CQI_MEAN']]
base.rename(columns={'PRB_AVAIL': 'BW_MHz'}, inplace=True)

### Deletar Variáveis não mais utilizadas

In [11]:
import gc # Garbage Colector
del [basebw, basekpi, baseocup, spazio]
gc.collect()

20455

### Identificação dos Setores

In [12]:
base.sort_values(['Endereço ID', 'Banda', 'Célula'], ascending=[True, True, True])
base.rename(columns={'BTS/NodeB/ENodeB': 'Site'}, inplace=True)

In [13]:
base['Cell'] = base.Célula.apply(lambda x: x[-2])
base.loc[(base['Cell'] == 'P') , 'Banda']='2600P'
base['Cell'] = base.Célula.apply(lambda x: x[-1])
base.loc[(base['Cell'] == 'A') | (base['Cell'] == 'E') | (base['Cell'] == 'I') | (base['Cell'] == 'M') | (base['Cell'] == 'Q') | (base['Cell'] == '1') | (base['Cell'] == 'X'), 'Setor']='1'
base.loc[(base['Cell'] == 'B') | (base['Cell'] == 'F') | (base['Cell'] == 'J') | (base['Cell'] == 'N') | (base['Cell'] == 'R') | (base['Cell'] == '2') | (base['Cell'] == 'Y'), 'Setor']='2'
base.loc[(base['Cell'] == 'C') | (base['Cell'] == 'G') | (base['Cell'] == 'K') | (base['Cell'] == 'O') | (base['Cell'] == 'S') | (base['Cell'] == '3') | (base['Cell'] == 'Z'), 'Setor']='3'
base.loc[(base['Cell'] == 'D') | (base['Cell'] == 'H') | (base['Cell'] == 'L') | (base['Cell'] == 'P') | (base['Cell'] == 'T') | (base['Cell'] == '4') | (base['Cell'] == 'W'), 'Setor']='4'

### SEPARANDO BASES POR CLASSIFICAÇÃO

In [14]:
cellbom = base[base.Classificação.eq('Bom')]
cellalerta = base[base.Classificação.eq('Alerta')]
cellcritico = base[base.Classificação.eq('Crítico')]

## OPORTUNIDADE DE BALANCEAMENTO BOM VS CRITICA

In [16]:
base.Classificação.value_counts()

Bom        22186
Alerta      1470
Crítico      673
Name: Classificação, dtype: int64

In [18]:
base.Classificação.eq('Bom').value_counts()

True     22186
False     2143
Name: Classificação, dtype: int64

## OPORTUNIDADE DE BALANCEAMENTO BOM VS ALERTA

## FATOR DE CRITICIDADE DA CIDADE